In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

ff = pd.read_csv("./data/F-F_Research_Data_Factors_daily.csv")

ff = ff.rename(columns={
    'TIME': "Date",
    "Mkt-RF": "MKT",
    "SMB": "SMB",
    "HML": "HML",
    "RF": "RF"
})

ff = ff.dropna() # 删除有缺失值的行

ff[['MKT','SMB','HML','RF']] = ff[['MKT','SMB','HML','RF']] / 100

ff['Date'] = pd.to_datetime(ff['Date'], format='%Y%m%d')


In [8]:
port = pd.read_csv("./data/6_Portfolios_2x3_Daily.csv")

port = port.rename(columns={
    'TIME': "Date"
})

port = port.dropna() # 删除空行

value_cols = [
    "SMALL LoBM",
    "ME1 BM2",
    "SMALL HiBM",
    "BIG LoBM",
    "ME2 BM2",
    "BIG HiBM"
]

port[value_cols] = port[value_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

port['Date'] = pd.to_datetime(port['Date'], format='%Y%m%d')
port = port.set_index('Date')

port[value_cols] = port[value_cols] / 100

In [9]:
data = port.merge(ff, left_index=True, right_on='Date')
data = data.set_index('Date')

In [10]:
port_cols = port.columns

for c in port_cols:
    data[c] = data[c] - data['RF']

In [11]:
results = {}

X = data[['MKT','SMB','HML']]
X = sm.add_constant(X)

In [12]:
for col in port_cols:
    y = data[col]
    
    model = sm.OLS(y, X).fit()
    
    results[col] = {
        "alpha": model.params['const'],
        "t_alpha": model.tvalues['const'],
        "R2": model.rsquared
    }

In [13]:
table = pd.DataFrame(results).T
print(table)

                alpha     t_alpha        R2
SMALL LoBM   1.885207  161.193848  0.000012
ME1 BM2      2.053328  177.595836  0.000007
SMALL HiBM   2.096235  160.555205  0.000001
BIG LoBM    23.757548   79.148077  0.000010
ME2 BM2     13.689662   93.764558  0.000025
BIG HiBM    12.255465   86.459588  0.000031


In [14]:
factor_stats = data[['MKT','SMB','HML']].agg(['mean','std'])

factor_stats.loc['t-stat'] = (
    factor_stats.loc['mean'] /
    factor_stats.loc['std'] *
    np.sqrt(len(data))
)

print(factor_stats)

             MKT       SMB       HML
mean    0.000307  0.000039  0.000152
std     0.010780  0.005941  0.006265
t-stat  9.208414  2.104293  7.834408


In [17]:
# 诊断：看ff读进来后的样子
print(ff.head(10))
print("---")
print(ff.dtypes)
print("---")
print(f"总行数: {len(ff)}")

        Date     MKT     SMB     HML      RF
0 1926-07-01  0.0009 -0.0025 -0.0027  0.0001
1 1926-07-02  0.0045 -0.0033 -0.0006  0.0001
2 1926-07-06  0.0017  0.0030 -0.0039  0.0001
3 1926-07-07  0.0009 -0.0058  0.0002  0.0001
4 1926-07-08  0.0022 -0.0038  0.0019  0.0001
5 1926-07-09 -0.0071  0.0043  0.0057  0.0001
6 1926-07-10  0.0061 -0.0053 -0.0010  0.0001
7 1926-07-12  0.0004 -0.0003  0.0064  0.0001
8 1926-07-13  0.0048 -0.0028 -0.0020  0.0001
9 1926-07-14  0.0004  0.0007 -0.0043  0.0001
---
Date    datetime64[us]
MKT            float64
SMB            float64
HML            float64
RF             float64
dtype: object
---
总行数: 26212


In [18]:
print(ff.columns.tolist())

['Date', 'MKT', 'SMB', 'HML', 'RF']
